In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os
import sys
import pandas as pd
import numpy as np

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset

# 프로젝트 디렉토리 경로 설정
project_dir = os.path.abspath("../")
sys.path.insert(0, project_dir)

# 필요한 모듈 불러오기
from lib.utils.confing import load_config
from lib.utils.data import prepare_dataloaders
from lib.model.hbt import (
    create_proposed_model,
    create_benchmark_model,
    HierarchicalModelTrainer,
)

In [2]:
# 설정 로드
config = load_config("inference")

# 경로 설정
model_type = (
    "proposed"
    if config["model"][config["model"]["select"]]["benchmark"]["enabled"] == False
    else "benchmark"
)
filename = (
    config["model"]["hbt"]["save_path"]["proposed"]
    if model_type == "proposed"
    else config["model"]["hbt"]["save_path"]["benchmark"]
)
model_path = config["model"]["hbt"]["save_path"]["path"] + filename
data_dir = f"{project_dir}/data/binance/futures/um/dataset/"

# 테스트 데이터 경로
retail_test_path = os.path.join(
    data_dir, f"retail_test_{config['dataloader']['timeframe']}.csv"
)
institutional_test_path = os.path.join(
    data_dir, f"institutional_test_{config['dataloader']['timeframe']}.csv"
)

print(f"모델 경로: {model_path}")
print(
    f"테스트 데이터 경로: \n- 소매: {retail_test_path}\n- 기관: {institutional_test_path}"
)


# 테스트 데이터 로드
def load_test_data(retail_path, institutional_path):
    print(f"소매 투자자 테스트 데이터 로드: {retail_path}")
    retail_df = pd.read_csv(retail_path, index_col=0)

    print(f"기관 투자자 테스트 데이터 로드: {institutional_path}")
    institutional_df = pd.read_csv(institutional_path, index_col=0)

    # 데이터 기본 정보 확인
    print(f"소매 투자자 데이터 크기: {retail_df.shape}")
    print(f"기관 투자자 데이터 크기: {institutional_df.shape}")

    # NaN 값 처리
    retail_df = retail_df.interpolate(method="linear", axis=0, limit_direction="both")
    retail_df = retail_df.fillna(method="ffill").fillna(method="bfill")

    institutional_df = institutional_df.interpolate(
        method="linear", axis=0, limit_direction="both"
    )
    institutional_df = institutional_df.fillna(method="ffill").fillna(method="bfill")

    print("데이터 로드 완료")
    return retail_df, institutional_df


# 테스트 데이터 로드
retail_df, institutional_df = load_test_data(retail_test_path, institutional_test_path)

# 데이터셋 준비
data_sources = {
    "retail": (retail_df, institutional_df),
    "institutional": (institutional_df, retail_df),
}

# 테스트 데이터 로더 준비
loaders = prepare_dataloaders(data_sources, config)

# 학습 데이터셋 통합
combined_train_dataset = ConcatDataset(
    [loaders["retail"]["train"].dataset, loaders["institutional"]["train"].dataset]
)
combined_train_loader = DataLoader(
    combined_train_dataset,
    batch_size=config["model"]["batch_size"],
    shuffle=True,
)

모델 경로: /home/youngjins/project/belief_trading/log/inference/2025-04-14 00:36:31/best_hbt_model.pth
테스트 데이터 경로: 
- 소매: /home/youngjins/project/belief_trading/data/binance/futures/um/dataset/retail_test_15m.csv
- 기관: /home/youngjins/project/belief_trading/data/binance/futures/um/dataset/institutional_test_15m.csv
소매 투자자 테스트 데이터 로드: /home/youngjins/project/belief_trading/data/binance/futures/um/dataset/retail_test_15m.csv
기관 투자자 테스트 데이터 로드: /home/youngjins/project/belief_trading/data/binance/futures/um/dataset/institutional_test_15m.csv
소매 투자자 데이터 크기: (9936, 109)
기관 투자자 데이터 크기: (9936, 109)
데이터 로드 완료


KeyError: 'train'

In [4]:
# 모델 파라미터 추출
model_config = config["model"]["hbt"]
ohlcv_dim = config["dataloader"]["ohlcv_dim"]
action_dim = model_config["action_dim"]
hidden_dim = model_config["hidden_dim"]
context_length = model_config["context_length"]
device = config["model"]["device"]

# 모델 유형 선택 (proposed 또는 benchmark)
model_type = "proposed"  # 또는 "benchmark"

print(f"모델 타입: {model_type}")
print(f"OHLCV 차원: {ohlcv_dim}")
print(f"액션 차원: {action_dim}")
print(f"히든 차원: {hidden_dim}")
print(f"컨텍스트 길이: {context_length}")
print(f"장치: {device}")

# 모델 생성
if model_type == "proposed":
    config["model"]["hbt"]["use_other_player_beliefs"] = True
    config["model"]["hbt"]["benchmark"]["enabled"] = False
    model = create_proposed_model(
        ohlcv_dim=ohlcv_dim,
        action_dim=action_dim,
        hidden_dim=hidden_dim,
        context_length=context_length,
    )
    trainer = HierarchicalModelTrainer(model=model)
else:
    config["model"]["hbt"]["use_other_player_beliefs"] = False
    config["model"]["hbt"]["benchmark"]["enabled"] = True
    model = create_benchmark_model(
        ohlcv_dim=ohlcv_dim,
        action_dim=action_dim,
        hidden_dim=hidden_dim,
        context_length=context_length,
    )
    trainer = HierarchicalModelTrainer(model=model)

# 모델 로드
trainer.load_model(model_path)
model = trainer.model.to(device)
model.eval()
print(f"모델을 {device} 장치로 이동했습니다")
print(f"모델 로드 완료: {model_path}")

모델 타입: proposed
OHLCV 차원: 9
액션 차원: 100
히든 차원: 512
컨텍스트 길이: 240
장치: cuda:3
모델을 cuda:3 장치로 이동했습니다
모델 로드 완료: /home/youngjins/project/belief_trading/log/inference/2025-04-14 00:36:31/best_hbt_model.pth


In [6]:
def run_inference(model, data_loader, device):
    model.eval()
    all_predictions = []
    all_targets = []
    all_losses = []
    
    with torch.no_grad():
        for batch in data_loader:
            x_ohlcv = batch["ohlcv"].to(device)
            x_self_actions = batch["self_actions"].to(device)
            x_other_actions = batch["other_actions"].to(device)
            y_target = batch["target"].to(device)
            
            # 예측 및 손실 계산            
            outputs, _ = model(x_ohlcv, x_self_actions, x_other_actions)

            # loss 계산
            loss = F.kl_div(
                torch.log(predictions + 1e-10),
                y_target,
                reduction="batchmean",
                log_target=False,
            )
            
            all_predictions.append(outputs.cpu().numpy())
            all_targets.append(y_target.cpu().numpy())
            all_losses.append(loss.item())
    
    # 결과 결합
    predictions = np.vstack(all_predictions)
    targets = np.vstack(all_targets)
    avg_loss = sum(all_losses) / len(all_losses)
    
    return predictions, targets, avg_loss

# 소매 및 기관 투자자 데이터에 대한 추론 수행
results = {}

for investor_type in ["retail", "institutional"]:
    print(f"{investor_type} 투자자 데이터에 대한 추론 수행 중...")
    test_loader = combined_train_loader[investor_type]["test"]
    predictions, targets, avg_loss = run_inference(model, test_loader, device)
    
    results[investor_type] = {
        "predictions": predictions,
        "targets": targets,
        "avg_loss": avg_loss
    }
    
    print(f"{investor_type} 추론 완료 - 평균 손실: {avg_loss:.4f}")

retail 투자자 데이터에 대한 추론 수행 중...


NameError: name 'combined_train_loader' is not defined

In [5]:
loaders

{'retail': {'test': <torch.utils.data.dataloader.DataLoader at 0x7f8413501100>},
 'institutional': {'test': <torch.utils.data.dataloader.DataLoader at 0x7f8413524be0>}}